# 03b · Label efficiency re-scored on de-duplicated test sets (CPU only)

Closes the last audit gap: label-efficiency probes were scored only on full test sets because per-image predictions were not saved.

This notebook:
1. Reads the **cached frozen CLS features** (`features_v2/<backbone>/<target>_{train,test}.npz`). No images, no GPU.
2. Refits the same probes: 20 class-stratified subsets per fraction, fixed C = 0.316, training split only, no refit.
3. **Checks every refit against the stored accuracy** in `03_label_efficiency_runs`. A different CPU or library build can flip a few borderline test images (one rice image = 0.068 pp), so it records every difference and stops only above 1 pp, which would mean wrong features. At the end, it also checks the full-test-set tests against the published ones.
4. Saves per-image predictions to `results_v2/correct/le/`.
5. Re-scores on the drop sets of `14_drop_sets.json` (`none`, `cos0.95`, `cos0.95+phash8`), using the paired t-test and Holm logic of notebook 03.

**Runtime:** RiceLeaf finishes in minutes. All four targets take about 1–2 h on a standard CPU runtime, mostly CUB. Results are saved after every backbone × target, so you can re-run after a disconnect and it resumes.

Run all, then download **`03b_label_efficiency_dedup.json`** from `results_v2/` and send it back.

In [ ]:
import os, sys, json, time, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
BASE = os.environ.get('SSL_BASE', '/content/drive/MyDrive/SSL_AgriStudy')
R, FEATS = f'{BASE}/results_v2', f'{BASE}/features_v2'
OUT_C = f'{R}/correct/le'; os.makedirs(OUT_C, exist_ok=True)
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_rel

TARGETS   = ['RiceLeaf', 'FloraBD', 'Flowers102', 'CUB']   # RiceLeaf first: the gap named in the paper
BACKBONES = ['imagenet_dino', 'florabd_full_scratch', 'florabd_cont_in', 'rice_full', 'rice_cont_in', 'dinov2_vits14']
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS, C_FIXED = list(range(20)), 0.316
DROPS = ['none', 'cos0.95', 'cos0.95+phash8']
drop_sets = json.load(open(f'{R}/14_drop_sets.json'))

# stored accuracies from notebook 03 (one JSON per run)
stored = {}
for f in glob.glob(f'{R}/03_label_efficiency_runs/*.json'):
    r = json.load(open(f))['record']; stored[(r['backbone'], r['target'], r['fraction'], r['seed'])] = r
print(len(stored), 'stored runs')
for t in TARGETS:
    for bb in BACKBONES:
        for sp in ('train', 'test'):
            assert os.path.exists(f'{FEATS}/{bb}/{t}_{sp}.npz'), f'missing feature cache {bb}/{t}_{sp}'
print('feature caches present')

In [ ]:
def stratified_subset(labels, fraction, seed, min_per_class=1):   # identical to sslcommon.py
    rng = np.random.RandomState(seed); idx, per = [], []
    for c in np.unique(labels):
        ci = np.where(labels == c)[0]; rng.shuffle(ci)
        n = min(len(ci), max(min_per_class, int(round(len(ci) * fraction))))
        idx.extend(ci[:n].tolist()); per.append(n)
    return np.array(sorted(idx)), per

def probe(Xtr, ytr, Xte):                                         # linear_probe(..., Cs=[0.316], refit_trainval=False)
    sc = StandardScaler().fit(Xtr)
    return LogisticRegression(C=C_FIXED, max_iter=3000).fit(sc.transform(Xtr), ytr).predict(sc.transform(Xte))

for t in TARGETS:
    for bb in BACKBONES:
        f_out = f'{OUT_C}/{bb}__{t}.npz'
        if os.path.exists(f_out): print(f'{bb:22s} {t:10s} cached'); continue
        ztr, zte = np.load(f'{FEATS}/{bb}/{t}_train.npz'), np.load(f'{FEATS}/{bb}/{t}_test.npz')
        Xtr, ytr, Xte, yte = ztr['X'], ztr['y'], zte['X'], zte['y']
        nmax = max([max(v) for v in drop_sets[t].values() if v] + [0])
        assert nmax < len(yte), f'{t}: drop-set index {nmax} outside test set of {len(yte)}'
        t0, preds, keys, diffs = time.time(), [], [], []
        for frac in FRACTIONS:
            for seed in (SEEDS if frac < 1 else [0]):
                idx, _ = stratified_subset(ytr, frac, seed)
                p = probe(Xtr[idx], ytr[idx], Xte); acc = 100 * (p == yte).mean()
                ref = stored.get((bb, t, frac, seed))
                assert ref is not None, f'no stored run for {bb} {t} {frac} s{seed}'
                assert ref['n_train'] == len(idx), f'subset size differs: {bb} {t} {frac} s{seed}'
                diffs.append(acc - ref['test_acc'])
                preds.append(p.astype(np.int16)); keys.append(f'{frac}__s{seed}')
        diffs = np.array(diffs); worst = float(np.abs(diffs).max())
        # lbfgs on another CPU/BLAS/sklearn build can flip a few borderline test images (1 rice image = 0.068 pp).
        # Wrong or stale features would shift accuracy by whole points, so stop only above 1 pp.
        assert worst < 1.0, f'{bb} {t}: reproduction differs from stored accuracy by {worst:.3f} points: check features'
        np.savez(f_out, pred=np.stack(preds), y=yte, keys=np.array(keys), max_abs_diff=worst,
                 mean_abs_diff=float(np.abs(diffs).mean()), mean_signed_diff=float(diffs.mean()),
                 n_exact=int((np.abs(diffs) < 1e-9).sum()))
        print(f'{bb:22s} {t:10s} {len(keys)} probes: {int((np.abs(diffs) < 1e-9).sum())} exact, '
              f'mean |Δ| = {np.abs(diffs).mean():.3f}, max |Δ| = {worst:.3f} pp vs stored, {time.time()-t0:.0f}s')

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for i, j in enumerate(o): run = max(run, (m - i) * p[j]); adj[j] = min(1.0, run)
    return adj

rows = []
for t in TARGETS:
    for bb in BACKBONES:
        z = np.load(f'{OUT_C}/{bb}__{t}.npz'); y = z['y']
        for d in DROPS:
            keep = np.setdiff1d(np.arange(len(y)), np.array(drop_sets[t][d], int))
            for k, p in zip(z['keys'], z['pred']):
                frac, seed = k.split('__s')
                rows.append(dict(target=t, backbone=bb, dropset=d, fraction=float(frac), seed=int(seed),
                                 n_test=len(keep), acc=100 * (p[keep] == y[keep]).mean(),
                                 n_train=stored[(bb, t, float(frac), int(seed))]['n_train']))
res = pd.DataFrame(rows)

# fractions whose subsets equal a smaller fraction's are tested once (same rule as notebook 03)
dup = set()
for t, g in res[(res.backbone == 'imagenet_dino') & (res.dropset == 'none')].groupby('target'):
    seen = {}
    for frac in sorted(g.fraction.unique()):
        sig = tuple(g[g.fraction == frac].sort_values('seed').n_train)
        if sig in seen: dup.add((t, frac))
        else: seen[sig] = frac

out = {'n_test': {t: {d: int(res[(res.target == t) & (res.dropset == d)].n_test.iloc[0]) for d in DROPS} for t in TARGETS},
       'reproduction_vs_stored': {f'{bb}__{t}': {k: float(np.load(f'{OUT_C}/{bb}__{t}.npz')[k]) if k in np.load(f'{OUT_C}/{bb}__{t}.npz').files else None
                                  for k in ('max_abs_diff', 'mean_abs_diff', 'mean_signed_diff', 'n_exact')}
                                  for t in TARGETS for bb in BACKBONES},
       'sklearn': __import__('sklearn').__version__, 'numpy': np.__version__,
       'means': {}, 'tests': {}}
for d in DROPS:
    g0 = res[res.dropset == d]
    out['means'][d] = (g0.groupby(['target', 'fraction', 'backbone']).acc.agg(['mean', 'std']).reset_index()
                         .to_dict('records'))
    tests = []
    for (t, frac), g in g0[g0.fraction < 1].groupby(['target', 'fraction']):
        if (t, frac) in dup: continue
        ref = g[g.backbone == 'imagenet_dino'].set_index('seed').acc
        for bb in BACKBONES[1:]:
            cur = g[g.backbone == bb].set_index('seed').acc
            p = ttest_rel(cur, ref).pvalue; p = 1.0 if np.isnan(p) else float(p)
            tests.append(dict(target=t, fraction=frac, backbone=bb, mean_diff=float((cur - ref).mean()),
                              std_diff=float((cur - ref).std(ddof=1)), p_paired_t=p))
    for r, pa in zip(tests, holm([r['p_paired_t'] for r in tests])): r['p_holm'] = float(pa)
    out['tests'][d] = tests
    print(f'{d}: {len(tests)} tests in the Holm family')

# check: full-test-set tests must reproduce the published ones (03_label_efficiency_tests.json)
pub_f = f'{R}/03_label_efficiency_tests.json'
if os.path.exists(pub_f):
    pub = {(r['target'], r['fraction'], r['backbone']): r for r in json.load(open(pub_f))}
    mine = {(r['target'], r['fraction'], r['backbone']): r for r in out['tests']['none']}
    if set(pub) == set(mine):
        dd = max(abs(pub[k]['mean_diff'] - mine[k]['mean_diff']) for k in pub)
        dp = max(abs(pub[k]['p_holm'] - mine[k]['p_holm']) for k in pub)
        print(f'published tests reproduced: {len(pub)} tests, max |Δ mean_diff| = {dd:.2e}, max |Δ p_holm| = {dp:.2e}')
        out['reproduces_published'] = dict(n=len(pub), max_diff_mean=dd, max_diff_pholm=dp)
    else:
        print('⚠ test set differs from the published family:', len(pub), 'published vs', len(mine), 'here')
json.dump(out, open(f'{R}/03b_label_efficiency_dedup.json', 'w'), indent=1)
print('saved', f'{R}/03b_label_efficiency_dedup.json')

In [ ]:
# quick look: domain checkpoints vs ImageNet DINO, full vs de-duplicated test set
T = {d: pd.DataFrame(out['tests'][d]) for d in DROPS}
v = T['none'][['target', 'fraction', 'backbone', 'mean_diff', 'p_holm']].rename(columns={'mean_diff': 'Δ full', 'p_holm': 'pHolm full'})
for d in DROPS[1:]:
    v[f'Δ {d}'] = T[d].mean_diff.values; v[f'pHolm {d}'] = T[d].p_holm.values
sig_pos = {d: int(((T[d].mean_diff > 0) & (T[d].p_holm < 0.05)).sum()) for d in DROPS}
print('significant advantages over ImageNet DINO:', sig_pos)
display(v.round(3))

In [ ]:
if IN_COLAB:
    from google.colab import drive; drive.flush_and_unmount(); print('✓ uploaded to Drive')